#**Projeto Cronos**
##CHALLENGE LOCAWEB 2026
Integrantes:

* Bruno Rosa - RM563779
* Danilo Alves - RM564109
* Enzo Cremaschi - RM562058
* Vinícius Macedo - RM561911

# 03 — Camada Gold | Projeto Cronos (Locaweb Challenge 2026)


**Duas limitações aceitas:**
- Não testamos o efeito de dezembro na série de volume — só temos 1 dezembro completo em 2025, então qualquer teste teria poder estatístico quase nulo (mesma lógica da limitação de 52 semanas).
- Não testamos a taxa de violação por `Status` dentro do universo elegível — `Status` já está fora das features do Desafio 3 por ser pós-evento (vazamento), então o teste não mudaria nenhuma decisão de engenharia, mesmo que confirmasse uma associação real.

**Contrato desta camada:** nenhum encoding aprendido de categórica (Ordered Target Statistics, TF-IDF, frequência baseada no alvo) é calculado aqui. E, novidade explícita desta versão: **nenhuma estatística baseada na variável-alvo** (`KPI Violado?`) é pré-calculada por grupo aqui — nem mesmo com shrinkage. Isso inclui explicitamente qualquer forma de "taxa de violação por Grupo designado" — esse cálculo só pode acontecer dentro do fold de treino, no notebook de modelagem do Desafio 3.

**Pré-requisito:** `02_silver_limpeza_texto.ipynb` já executado.


In [21]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [22]:
import sys
sys.path.append('/content/drive/MyDrive/cronos_project')

import pandas as pd
import numpy as np

from utils import (
    setup_logging,
    PROJECT_PATHS,
    load_parquet_layer,
    save_parquet_with_metadata,
    profile_dataframe,
    add_cyclical_time_features,
    expanding_count_by_key,
    hours_since_last_event_by_key,
    rolling_window_causal_count,
    duration_reference_stats_by_group,
    flag_low_volume_groups,
)

logger = setup_logging('gold')
PROJECT_PATHS.ensure_dirs()

In [23]:
silver_path = PROJECT_PATHS.silver / 'lw_incidentes_clean.parquet'
df = load_parquet_layer(silver_path, logger=logger)
df['data'] = df['Aberto'].dt.date
df.shape

2026-08-16 13:17:16 | INFO     | gold | Lido de /content/drive/MyDrive/cronos_project/data/silver/lw_incidentes_clean.parquet: 121811 linhas x 23 colunas (5 colunas de metadado: ['_ingested_at', '_source_layer', '_source_hash', '_flag_encerrado_antes_aberto', '_flag_resolvido_apos_encerrado']).


INFO:gold:Lido de /content/drive/MyDrive/cronos_project/data/silver/lw_incidentes_clean.parquet: 121811 linhas x 23 colunas (5 colunas de metadado: ['_ingested_at', '_source_layer', '_source_hash', '_flag_encerrado_antes_aberto', '_flag_resolvido_apos_encerrado']).


(121811, 29)

---
## PARTE 1 — Motor de Features Gerais

Esta seção constrói `df_features`, uma versão enriquecida da Silver com todas as colunas causais que fazem sentido em mais de um desafio. Nenhuma linha é removida ou filtrada aqui — filtros de elegibilidade (ex.: `Entrou para KPI? == SIM`) acontecem só na tabela de destino de cada desafio, na Parte 3/4.


### 1.1 Features de calendário (usadas no Desafio 1 agregado E no Desafio 3 por ticket)

In [24]:
df_features = df.copy()

df_features['dia_semana'] = df_features['Aberto'].dt.dayofweek
df_features['fim_de_semana'] = (df_features['dia_semana'] >= 5).astype(int)
df_features['hora_abertura'] = df_features['Aberto'].dt.hour
df_features['dia_do_mes'] = df_features['Aberto'].dt.day
df_features['mes'] = df_features['Aberto'].dt.month

df_features = add_cyclical_time_features(df_features, df_features['dia_semana'], period=7, prefix='dow')
df_features = add_cyclical_time_features(df_features, df_features['hora_abertura'], period=24, prefix='hora')

try:
    import holidays
    feriados_br = holidays.Brazil(years=[2025])
except ImportError:
    logger.warning('Pacote "holidays" não instalado — rode !pip install holidays. feriados ficará vazio.')
    feriados_br = {}

feriados_dates = set(pd.to_datetime(list(feriados_br.keys())).date) if feriados_br else set()
datas_dt = pd.to_datetime(df_features['data'])

df_features['feriado'] = datas_dt.dt.date.isin(feriados_dates).astype(int)
df_features['vespera_feriado'] = (datas_dt + pd.Timedelta(days=1)).dt.date.isin(feriados_dates).astype(int)
df_features['dia_seguinte_feriado'] = (datas_dt - pd.Timedelta(days=1)).dt.date.isin(feriados_dates).astype(int)

def _turno(hora):
    if 0 <= hora < 6:
        return 'madrugada'
    elif 6 <= hora < 18:
        return 'comercial'
    return 'noite'

df_features['turno_abertura'] = df_features['hora_abertura'].apply(_turno)
df_features['fora_horario_comercial'] = (df_features['turno_abertura'] != 'comercial').astype(int)

logger.info('Features de calendário: OK (%d colunas adicionadas).', 11)

2026-08-16 13:17:16 | INFO     | gold | Features de calendário: OK (11 colunas adicionadas).


INFO:gold:Features de calendário: OK (11 colunas adicionadas).


### 1.2 Features de texto (verbosidade e recorrência — usadas no Desafio 2 como complemento e no Desafio 3)

In [25]:
df_features['descricao_n_tokens'] = df_features['descricao_limpa'].fillna('').str.split().str.len()

df_features['descricao_contagem_historica'] = expanding_count_by_key(
    df_features, key_col='descricao_limpa', time_col='Aberto', new_col='descricao_contagem_historica'
)

logger.info('Features de texto: OK.')

2026-08-16 13:17:17 | INFO     | gold | Features de texto: OK.


INFO:gold:Features de texto: OK.


### 1.3 Features de recorrência causal (ativo, equipe, incidente pai)

Todas seguem a mesma disciplina: contam apenas o que já aconteceu ANTES do timestamp da linha atual — nunca o presente ou o futuro. Testado explicitamente na suite `tests/test_utils.py`.

In [26]:
df_features['ic_contagem_historica'] = expanding_count_by_key(
    df_features, key_col='Item de configuração', time_col='Aberto', new_col='ic_contagem_historica'
)
df_features['ic_horas_desde_ultimo_chamado'] = hours_since_last_event_by_key(
    df_features, key_col='Item de configuração', time_col='Aberto', new_col='ic_horas_desde_ultimo_chamado'
)

df_features['grupo_contagem_historica'] = expanding_count_by_key(
    df_features, key_col='Grupo designado', time_col='Aberto', new_col='grupo_contagem_historica'
)
df_features['grupo_chamados_ultima_hora'] = rolling_window_causal_count(
    df_features, key_col='Grupo designado', time_col='Aberto', window='60min', new_col='grupo_chamados_ultima_hora'
)

df_features['tem_incidente_pai'] = df_features['Incidente Pai'].notna().astype(int)
df_features['incidente_pai_contagem_historica'] = expanding_count_by_key(
    df_features, key_col='Incidente Pai', time_col='Aberto', new_col='incidente_pai_contagem_historica'
)

logger.info('Features de recorrência causal: OK.')

2026-08-16 13:17:17 | INFO     | gold | Features de recorrência causal: OK.


INFO:gold:Features de recorrência causal: OK.


### 1.4 Flag de baixo volume amostral

**Evidência:** a taxa de violação por `Grupo designado` varia de 0% a 10,26% — mas equipes como Team04 (n=16), Team06 (n=39), Team01 (n=41) e Team15 (n=50) têm volume elegível tão baixo que a taxa observada tem alta variância amostral (um único evento a mais ou a menos muda a taxa em vários pontos percentuais).

**Decisão:** adicionar uma flag booleana de baixo volume, calculada só a partir de CONTAGEM (nunca da variável-alvo) — por isso é segura de pré-calcular aqui, ao contrário de um shrinkage bayesiano de taxa de violação, que usaria o alvo e só pode ser calculado dentro do fold de treino no notebook de modelagem.

In [27]:
df_features['grupo_baixo_volume'] = flag_low_volume_groups(df_features, 'Grupo designado', min_volume=100)

logger.info(
    '%.1f%% dos chamados pertencem a um Grupo designado de baixo volume (<100 registros totais).',
    df_features['grupo_baixo_volume'].mean() * 100,
)

2026-08-16 13:17:17 | INFO     | gold | 0.0% dos chamados pertencem a um Grupo designado de baixo volume (<100 registros totais).


INFO:gold:0.0% dos chamados pertencem a um Grupo designado de baixo volume (<100 registros totais).


### 1.5 Tabela de referência: Duração segmentada por Status

**Evidência:** Kruskal-Wallis (H=50.053, p≈0) confirmou que `Duração` é uma mistura de pelo menos 3 processos de fechamento diferentes. Esta tabela é gerada como **referência/diagnóstico** — não vira uma coluna por-ticket, porque `Duração` só é conhecida no fechamento e não pode ser feature de um modelo que prevê algo na abertura.

In [28]:
tabela_duracao_por_status = duration_reference_stats_by_group(df_features, 'Duração', ['Status'])
tabela_duracao_por_status

,mediana,media,p90,contagem
Status,,,,
Aguardando Problema,14537.0,14537.0,14537.0,1
Encerrado Automaticamente,7590.5,510281.4,596335.6,26174
Encerrado,3121.0,62633.1,21599.5,15266
Sem Intervenção,334.0,16667.8,3546.0,80370


In [29]:
tabela_duracao_por_status_prioridade = duration_reference_stats_by_group(
    df_features, 'Duração', ['Status', 'Prioridade']
)
tabela_duracao_por_status_prioridade

mediana     media        p90  \
Status                    Prioridade                                       
Encerrado Automaticamente 5 - Muito Baixa  594760.0  569438.1   861139.0   
Sem Intervenção           5 - Muito Baixa  129436.0  129436.0   129436.0   
Encerrado                 4 - Baixa        108485.0  894735.1  2410711.0   
                          1 - Crítica       31742.0   31742.0    31742.0   
Aguardando Problema       3 - Média         14537.0   14537.0    14537.0   
Encerrado Automaticamente 4 - Baixa         10827.0  173335.5   571505.7   
Encerrado                 3 - Média          6648.0  115422.0    74707.1   
Encerrado Automaticamente 3 - Média          6017.0  665345.0   601783.8   
Encerrado                 5 - Muito Baixa    3222.5   59935.4    90453.7   
                          2 - Alta           2117.0    3925.7     7505.9   
Sem Intervenção           2 - Alta           1279.5    2282.0     5622.8   
Encerrado Automaticamente 2 - Alta           1250.0    7950.2     8680.6   
Sem Intervenção           3 - Média           604.0    9536.7     4490.5   
                          4 - Baixa           300.0   20552.4     2716.0   

                                           contagem  
Status                    Prioridade                 
Encerrado Automaticamente 5 - Muito Baixa       282  
Sem Intervenção           5 - Muito Baixa         1  
Encerrado                 4 - Baixa             305  
                          1 - Crítica             1  
Aguardando Problema       3 - Média               1  
Encerrado Automaticamente 4 - Baixa            8134  
Encerrado                 3 - Média            5580  
Encerrado Automaticamente 3 - Média           17713  
Encerrado                 5 - Muito Baixa        42  
                          2 - Alta             9338  
Sem Intervenção           2 - Alta             6262  
Encerrado Automaticamente 2 - Alta               45  
Sem Intervenção           3 - Média           17966  
                          4 - Baixa           56141

### 1.6 Profiling da camada de features gerais

In [30]:
logger.info('df_features: %d linhas x %d colunas (Silver tinha %d colunas).', *df_features.shape, df.shape[1])
profile_dataframe(df_features.drop(columns=[c for c in df_features.columns if c.startswith('_')]))

2026-08-16 13:17:18 | INFO     | gold | df_features: 121811 linhas x 52 colunas (Silver tinha 29 colunas).


INFO:gold:df_features: 121811 linhas x 52 colunas (Silver tinha 29 colunas).


,dtype,n_nao_nulos,n_nulos,pct_nulos,n_unicos
Incidente Pai,object,15063,106748,87.63,3305
incidente_pai_contagem_historica,float64,15063,106748,87.63,630
Solução,object,15200,106611,87.52,2
KPI Violado?,object,25156,96655,79.35,2
Resolvido,datetime64[ns],39579,82232,67.51,36874
Código de fechamento,object,40086,81725,67.09,17
Produto,object,43883,77928,63.97,51
Categoria,object,44090,77721,63.80,140
Subcategoria,object,44091,77720,63.80,446
ic_horas_desde_ultimo_chamado,float64,110937,10874,8.93,53981


---
## PARTE 2 — Tabela Desafio 1: `gold_volume_diario` (SARIMAX)

Grão: 1 linha por dia (365 linhas, 2025 completo). Construída por agregação de `df_features`.

In [31]:
volume_diario = df_features.groupby('data').size().rename('qtde_chamados_total').to_frame()
volume_diario.index = pd.to_datetime(volume_diario.index)
volume_diario = volume_diario.asfreq('D').fillna(0).astype({'qtde_chamados_total': int})

mask_p2_p3 = df_features['Prioridade'].isin(['2 - Alta', '3 - Média'])
volume_p2_p3 = df_features[mask_p2_p3].groupby('data').size().rename('qtde_chamados_p2_p3')
volume_p2_p3.index = pd.to_datetime(volume_p2_p3.index)
volume_diario = volume_diario.join(volume_p2_p3, how='left').fillna({'qtde_chamados_p2_p3': 0})
volume_diario['qtde_chamados_p2_p3'] = volume_diario['qtde_chamados_p2_p3'].astype(int)

gold_volume = volume_diario.reset_index().rename(columns={'index': 'data'})

In [32]:
# Reaproveita as features de calendário já calculadas por ticket, agregadas ao grão diário
calendario_diario = (
    df_features[['data', 'dia_semana', 'fim_de_semana', 'feriado', 'vespera_feriado', 'dia_seguinte_feriado']]
    .assign(data=lambda d: pd.to_datetime(d['data']))
    .drop_duplicates(subset='data')
)
gold_volume = gold_volume.merge(calendario_diario, on='data', how='left')
gold_volume['dia_do_mes'] = gold_volume['data'].dt.day
gold_volume['mes'] = gold_volume['data'].dt.month
gold_volume = add_cyclical_time_features(gold_volume, gold_volume['dia_semana'], period=7, prefix='dow')

gold_volume.head()

,data,qtde_chamados_total,qtde_chamados_p2_p3,dia_semana,fim_de_semana,feriado,vespera_feriado,dia_seguinte_feriado,dia_do_mes,mes,dow_sin,dow_cos
0,2025-01-01,34,11,2,0,1,0,0,1,1,0.974928,-0.222521
1,2025-01-02,55,43,3,0,0,0,1,2,1,0.433884,-0.900969
2,2025-01-03,77,51,4,0,0,0,0,3,1,-0.433884,-0.900969
3,2025-01-04,166,149,5,1,0,0,0,4,1,-0.974928,-0.222521
4,2025-01-05,39,18,6,1,0,0,0,5,1,-0.781831,0.623490


In [33]:
# Lags operacionais: SEMPRE t-1 a t-7
for lag in range(1, 8):
    gold_volume[f'volume_lag_{lag}'] = gold_volume['qtde_chamados_total'].shift(lag)

gold_volume['media_movel_7d_ate_ontem'] = gold_volume['qtde_chamados_total'].shift(1).rolling(7).mean()

check_leak = (gold_volume['volume_lag_1'].iloc[1:].values == gold_volume['qtde_chamados_total'].iloc[:-1].values).all()
logger.info('Verificação de não-vazamento (volume_lag_1 == total de t-1): %s', check_leak)

2026-08-16 13:17:18 | INFO     | gold | Verificação de não-vazamento (volume_lag_1 == total de t-1): True


INFO:gold:Verificação de não-vazamento (volume_lag_1 == total de t-1): True


In [34]:
gold_volume_path = PROJECT_PATHS.gold / 'gold_volume_diario.parquet'
save_parquet_with_metadata(gold_volume, gold_volume_path, layer='gold', logger=logger)

2026-08-16 13:17:18 | INFO     | gold | Gravado: /content/drive/MyDrive/cronos_project/data/gold/gold_volume_diario.parquet (365 linhas, 0.0 MB).


INFO:gold:Gravado: /content/drive/MyDrive/cronos_project/data/gold/gold_volume_diario.parquet (365 linhas, 0.0 MB).


---
## PARTE 3 — Tabela Desafio 2: `gold_tendencias_macro` (diagnóstico)

Grão: 1 linha por semana × Prioridade × Categoria × Produto. Única tabela onde nulo de Categoria/Produto vira rótulo explícito (`"Não categorizado"`) — é visão de relatório, não feature de modelo.

In [35]:
df_macro = df_features.copy()
df_macro['semana'] = pd.to_datetime(df_macro['data']).dt.to_period('W-TUE').dt.start_time
df_macro['Categoria_rotulada'] = df_macro['Categoria'].fillna('Não categorizado')
df_macro['Produto_rotulado'] = df_macro['Produto'].fillna('Não categorizado')

gold_tendencias = (
    df_macro
    .groupby(['semana', 'Prioridade', 'Categoria_rotulada', 'Produto_rotulado'], observed=True)
    .agg(qtde_chamados=('Número', 'count'))
    .reset_index()
)

cobertura_semanal = df_macro.groupby('semana')['flag_categorizado'].mean().rename('pct_categorizado').reset_index()
gold_tendencias = gold_tendencias.merge(cobertura_semanal, on='semana', how='left')

gold_tendencias.head(10)

,semana,Prioridade,Categoria_rotulada,Produto_rotulado,qtde_chamados,pct_categorizado
0,2025-01-01,2 - Alta,Não categorizado,Não categorizado,123,0.778929
1,2025-01-01,2 - Alta,cat11,Não categorizado,1,0.778929
2,2025-01-01,2 - Alta,cat116,lsin,1,0.778929
3,2025-01-01,2 - Alta,cat121,lsin,1,0.778929
4,2025-01-01,2 - Alta,cat128,lsto,2,0.778929
5,2025-01-01,2 - Alta,cat132,lvmr,1,0.778929
6,2025-01-01,2 - Alta,cat137,lhco,1,0.778929
7,2025-01-01,2 - Alta,cat138,lhvp,1,0.778929
8,2025-01-01,2 - Alta,cat141,lsin,1,0.778929
9,2025-01-01,2 - Alta,cat17,lsin,1,0.778929


### 3.1 Enriquecimento com a tabela de referência de Duração

Junta a mediana de Duração por `Status` × `Prioridade` (Parte 1.5) como contexto de diagnóstico — só aqui, porque `gold_tendencias_macro` é uma tabela de relatório agregado, não uma feature de modelo por-ticket.

In [36]:
gold_tendencias_duracao = (
    df_macro.groupby(['semana', 'Prioridade'])['Duração']
    .median()
    .rename('duracao_mediana_semana')
    .reset_index()
)
gold_tendencias = gold_tendencias.merge(gold_tendencias_duracao, on=['semana', 'Prioridade'], how='left')

logger.info('gold_tendencias_macro: %d linhas.', len(gold_tendencias))
gold_tendencias_path = PROJECT_PATHS.gold / 'gold_tendencias_macro.parquet'
save_parquet_with_metadata(gold_tendencias, gold_tendencias_path, layer='gold', logger=logger)

2026-08-16 13:17:19 | INFO     | gold | gold_tendencias_macro: 6808 linhas.


INFO:gold:gold_tendencias_macro: 6808 linhas.


2026-08-16 13:17:19 | INFO     | gold | Gravado: /content/drive/MyDrive/cronos_project/data/gold/gold_tendencias_macro.parquet (6808 linhas, 0.0 MB).


INFO:gold:Gravado: /content/drive/MyDrive/cronos_project/data/gold/gold_tendencias_macro.parquet (6808 linhas, 0.0 MB).


---
## PARTE 4 — Tabela Desafios 3 e 4: `gold_chamados_risco` (CatBoost + SHAP)

Grão: 1 linha por chamado. Seleciona de `df_features` só as colunas seguras (exclui vazamento) e adiciona os alvos.

In [37]:
COLUNAS_VAZAMENTO = [
    'Resolvido', 'Encerrado', 'Duração', 'Código de fechamento', 'Solução', 'Status',
]

colunas_finais = [
    'Número', 'Aberto', 'data', 'Prioridade', 'Produto', 'Categoria', 'Subcategoria',
    'Grupo designado', 'Item de configuração', 'Aberto por', 'descricao_limpa',
    'flag_categorizado', 'Incidente Pai',
    'hora_abertura', 'dia_semana', 'fim_de_semana', 'hora_sin', 'hora_cos', 'dow_sin', 'dow_cos',
    'turno_abertura', 'fora_horario_comercial', 'vespera_feriado', 'dia_seguinte_feriado', 'feriado',
    'descricao_n_tokens', 'descricao_contagem_historica',
    'ic_contagem_historica', 'ic_horas_desde_ultimo_chamado',
    'grupo_contagem_historica', 'grupo_chamados_ultima_hora', 'grupo_baixo_volume',
    'tem_incidente_pai', 'incidente_pai_contagem_historica',
    'Entrou para KPI?', 'KPI Violado?',
]

# Feature contextual de pressão de fila, reaproveitada da Parte 2
contexto_volume = gold_volume[['data', 'media_movel_7d_ate_ontem']].rename(
    columns={'media_movel_7d_ate_ontem': 'pressao_fila_7d'}
)
df_features['data_join'] = pd.to_datetime(df_features['data'])
df_features = df_features.merge(
    contexto_volume.rename(columns={'data': 'data_join'}), on='data_join', how='left'
).drop(columns=['data_join'])

colunas_finais.append('pressao_fila_7d')
colunas_finais = [c for c in colunas_finais if c in df_features.columns]

gold_risco = df_features[colunas_finais].copy()
logger.info('gold_chamados_risco: %d linhas x %d colunas.', *gold_risco.shape)

2026-08-16 13:17:19 | INFO     | gold | gold_chamados_risco: 121811 linhas x 37 colunas.


INFO:gold:gold_chamados_risco: 121811 linhas x 37 colunas.


In [38]:
# Checagem explícita: nenhuma coluna de vazamento entrou na tabela final
assert set(COLUNAS_VAZAMENTO).isdisjoint(gold_risco.columns), 'Coluna de vazamento vazou para gold_chamados_risco!'
assert gold_risco['Número'].is_unique, 'gold_chamados_risco deveria ter 1 linha por chamado!'
logger.info('Checagem de vazamento: OK — nenhuma das colunas %s presente.', COLUNAS_VAZAMENTO)

gold_risco_path = PROJECT_PATHS.gold / 'gold_chamados_risco.parquet'
save_parquet_with_metadata(gold_risco, gold_risco_path, layer='gold', logger=logger)

2026-08-16 13:17:19 | INFO     | gold | Checagem de vazamento: OK — nenhuma das colunas ['Resolvido', 'Encerrado', 'Duração', 'Código de fechamento', 'Solução', 'Status'] presente.


INFO:gold:Checagem de vazamento: OK — nenhuma das colunas ['Resolvido', 'Encerrado', 'Duração', 'Código de fechamento', 'Solução', 'Status'] presente.


2026-08-16 13:17:19 | INFO     | gold | Gravado: /content/drive/MyDrive/cronos_project/data/gold/gold_chamados_risco.parquet (121811 linhas, 5.0 MB).


INFO:gold:Gravado: /content/drive/MyDrive/cronos_project/data/gold/gold_chamados_risco.parquet (121811 linhas, 5.0 MB).


---
## Checagens de sanidade finais (as três tabelas)

In [39]:
df_v = pd.read_parquet(gold_volume_path)
df_t = pd.read_parquet(gold_tendencias_path)
df_r = pd.read_parquet(gold_risco_path)

assert df_v.shape[0] == 365, f'esperado 365 dias de 2025, veio {df_v.shape[0]}'
assert set(COLUNAS_VAZAMENTO).isdisjoint(df_r.columns), 'Coluna de vazamento vazou!'
assert df_r['Número'].is_unique, 'gold_chamados_risco deveria ter 1 linha por chamado!'
assert 'grupo_baixo_volume' in df_r.columns, 'flag de baixo volume ausente!'
assert 'duracao_mediana_semana' in df_t.columns, 'referência de duração ausente na tabela de tendências!'

logger.info('Checagens de sanidade: OK nas três tabelas.')
logger.info('gold_volume_diario: %s | gold_tendencias_macro: %s | gold_chamados_risco: %s',
            df_v.shape, df_t.shape, df_r.shape)

2026-08-16 13:17:20 | INFO     | gold | Checagens de sanidade: OK nas três tabelas.


INFO:gold:Checagens de sanidade: OK nas três tabelas.


2026-08-16 13:17:20 | INFO     | gold | gold_volume_diario: (365, 22) | gold_tendencias_macro: (6808, 9) | gold_chamados_risco: (121811, 39)


INFO:gold:gold_volume_diario: (365, 22) | gold_tendencias_macro: (6808, 9) | gold_chamados_risco: (121811, 39)


## Resumo de execução

In [40]:
resumo_execucao = f'''
EXECUÇÃO DA CAMADA GOLD — {pd.Timestamp.now(tz="UTC").isoformat()}
Entrada (Silver): {df.shape[0]:,} linhas

Camada de features gerais (df_features): {df_features.shape[1]} colunas totais
  Novidades desta versão: grupo_baixo_volume (flag de instabilidade amostral),
  tabela de referência Duração x Status (não é feature de modelo, é diagnóstico)

gold_volume_diario.parquet: {df_v.shape[0]} linhas x {df_v.shape[1]} colunas
gold_tendencias_macro.parquet: {df_t.shape[0]:,} linhas x {df_t.shape[1]} colunas
gold_chamados_risco.parquet: {df_r.shape[0]:,} linhas x {df_r.shape[1]} colunas

Colunas excluídas de gold_chamados_risco por risco de vazamento: {COLUNAS_VAZAMENTO}

Limitações aceitas e documentadas (não resolvidas nesta versão):
  - Efeito de dezembro na série de volume: não testado (n=1 ano, poder estatístico insuficiente)
  - Taxa de violação por Status: não testada (Status já excluído das features por vazamento,
    teste não mudaria decisão de engenharia)
'''.strip()

print(resumo_execucao)

EXECUÇÃO DA CAMADA GOLD (v2) — 2026-08-16T13:17:20.047504+00:00
Entrada (Silver): 121,811 linhas

Camada de features gerais (df_features): 53 colunas totais
  Novidades desta versão: grupo_baixo_volume (flag de instabilidade amostral),
  tabela de referência Duração x Status (não é feature de modelo, é diagnóstico)

gold_volume_diario.parquet: 365 linhas x 22 colunas
gold_tendencias_macro.parquet: 6,808 linhas x 9 colunas
gold_chamados_risco.parquet: 121,811 linhas x 39 colunas

Colunas excluídas de gold_chamados_risco por risco de vazamento: ['Resolvido', 'Encerrado', 'Duração', 'Código de fechamento', 'Solução', 'Status']

Limitações aceitas e documentadas (não resolvidas nesta versão):
  - Efeito de dezembro na série de volume: não testado (n=1 ano, poder estatístico insuficiente)
  - Taxa de violação por Status: não testada (Status já excluído das features por vazamento,
    teste não mudaria decisão de engenharia)
